In [ ]:
# CELL 1 — Install & Koneksi DuckDB
import subprocess
subprocess.run(['pip', 'install', 'duckdb'], capture_output=True)

import duckdb
import pandas as pd

# Koneksi ke database DuckDB
# jika file tidak ada, DuckDB akan otomatis membuatnya saat kita melakukan operasi penulisan
con = duckdb.connect('../umkm_retail.duckdb')
print("DuckDB connected ✓")

DuckDB connected ✓


In [ ]:
# CELL 2 — Load CSV ke DuckDB Tables

con.execute("""
    CREATE OR REPLACE TABLE transactions AS
    SELECT * FROM read_csv_auto('../data/processed/transactions_clean.csv')
""")
con.execute("""
    CREATE OR REPLACE TABLE products AS
    SELECT * FROM read_csv_auto('../data/raw/products.csv')
""")
con.execute("""
    CREATE OR REPLACE TABLE customers AS
    SELECT * FROM read_csv_auto('../data/raw/customers.csv')
""")
con.execute("""
    CREATE OR REPLACE TABLE stores AS
    SELECT * FROM read_csv_auto('../data/raw/stores.csv')
""")

# Verifikasi
for tbl in ['transactions', 'products', 'customers', 'stores']:
    n = con.execute(f"SELECT COUNT(*) FROM {tbl}").fetchone()[0]
    print(f"✓ {tbl:15s} → {n:,} rows")

✓ transactions    → 5,414 rows
✓ products        → 56 rows
✓ customers       → 1,000 rows
✓ stores          → 50 rows


In [ ]:
# CELL 3 — Helper Function & Cek Schema

def query(sql):
    return con.execute(sql).df()

# Test schema
print(query("DESCRIBE transactions").to_string(index=False))

    column_name column_type null  key default extra
       order_id     VARCHAR  YES None    None  None
     order_date        DATE  YES None    None  None
    customer_id     VARCHAR  YES None    None  None
       store_id     VARCHAR  YES None    None  None
     product_id     VARCHAR  YES None    None  None
   product_name     VARCHAR  YES None    None  None
       category     VARCHAR  YES None    None  None
            qty      BIGINT  YES None    None  None
     unit_price      DOUBLE  YES None    None  None
   discount_pct      DOUBLE  YES None    None  None
discount_amount      DOUBLE  YES None    None  None
    total_price      DOUBLE  YES None    None  None
 payment_method     VARCHAR  YES None    None  None
   order_status     VARCHAR  YES None    None  None
           city     VARCHAR  YES None    None  None
           year      DOUBLE  YES None    None  None
          month      DOUBLE  YES None    None  None
        quarter      DOUBLE  YES None    None  None
     month_n

In [ ]:
# CELL 4 — Analisis 1: Revenue per Kategori per Kuartal

q1 = query("""
    SELECT
        YEAR(order_date::DATE)                    AS tahun,
        QUARTER(order_date::DATE)                 AS kuartal,
        category,
        COUNT(DISTINCT order_id)                  AS total_order,
        SUM(qty)                                  AS total_qty,
        ROUND(SUM(total_price), 0)                AS total_revenue,
        ROUND(AVG(total_price), 0)                AS avg_order_value
    FROM transactions
    WHERE order_status = 'delivered'
    GROUP BY 1, 2, 3
    ORDER BY 1, 2, total_revenue DESC
""")
print(q1.to_string(index=False))
q1.to_csv('../data/processed/q1_revenue_category_quarter.csv', index=False)
print("\n✓ Disimpan")

 tahun  kuartal                 category  total_order  total_qty  total_revenue  avg_order_value
  2023        1        Fashion & Pakaian           35       81.0     13731895.0         392340.0
  2023        1        Makanan & Minuman           57      132.0      9589300.0         168233.0
  2023        1     Kerajinan & Souvenir           40       93.0      9289810.0         232245.0
  2023        1    Elektronik & Aksesori           40       92.0      8500035.0         212501.0
  2023        1             Rumah Tangga           47      117.0      8174015.0         173915.0
  2023        1 Perlengkapan Bayi & Anak           33       69.0      6263600.0         189806.0
  2023        1   Kecantikan & Perawatan           45       99.0      4894390.0         108764.0
  2023        1       Olahraga & Outdoor           29       77.0      4862015.0         167656.0
  2023        2        Fashion & Pakaian           43       96.0     17345710.0         403389.0
  2023        2    Elektronik 

In [ ]:
# CELL 5 — Analisis 2: Dominasi Metode Pembayaran per Kota
q2 = query("""
    SELECT
        city,
        payment_method,
        COUNT(*)                                  AS total_transaksi,
        ROUND(SUM(total_price), 0)                AS total_revenue,
        ROUND(
            COUNT(*) * 100.0 /
            SUM(COUNT(*)) OVER (PARTITION BY city),
        2)                                        AS pct_per_kota
    FROM transactions
    WHERE order_status = 'delivered'
      AND payment_method != 'Unknown'
    GROUP BY city, payment_method
    ORDER BY city, total_transaksi DESC
""")
print(q2.to_string(index=False))
q2.to_csv('../data/processed/q2_payment_by_city.csv', index=False)
print("\n✓ Disimpan")

     city   payment_method  total_transaksi  total_revenue  pct_per_kota
  Bandung     Transfer Bca               27      4764750.0         12.05
  Bandung Transfer Mandiri               24      5589540.0         10.71
  Bandung              Cod               24      5312780.0         10.71
  Bandung        Shopeepay               22      3408485.0          9.82
  Bandung             Dana               20      3603175.0          8.93
  Bandung            Gopay               18      3556490.0          8.04
  Bandung     Transfer Bri               18      4910105.0          8.04
  Bandung             Qris               17      2649370.0          7.59
  Bandung              Ovo               16      2943650.0          7.14
  Bandung         Alfamart               15      2323130.0          6.70
  Bandung        Indomaret               14      1936845.0          6.25
  Bandung          Linkaja                9      1067050.0          4.02
    Bogor        Indomaret               34      53

In [ ]:
# CELL 6 — Analisis 3: Performa Toko (Top 20)
q3 = query("""
    WITH store_metrics AS (
        SELECT
            t.store_id,
            s.store_name,
            s.city,
            s.store_type,
            COUNT(DISTINCT t.order_id)       AS total_order,
            ROUND(SUM(t.total_price), 0)     AS total_revenue,
            ROUND(AVG(t.total_price), 0)     AS avg_order_value,
            COUNT(DISTINCT t.customer_id)    AS unique_customer,
            MAX(t.order_date::DATE)          AS last_order_date
        FROM transactions t
        JOIN stores s ON t.store_id = s.store_id
        WHERE t.order_status = 'delivered'
        GROUP BY 1, 2, 3, 4
    )
    SELECT
        *,
        NTILE(4) OVER (ORDER BY total_revenue) AS revenue_quartile
    FROM store_metrics
    ORDER BY total_revenue DESC
    LIMIT 20
""")
print(q3.to_string(index=False))
q3.to_csv('../data/processed/q3_store_performance.csv', index=False)
print("\n✓ Disimpan")

store_id               store_name      city store_type  total_order  total_revenue  avg_order_value  unique_customer last_order_date  revenue_quartile
  STR041     Toko Makmur Semarang  Semarang     Online           57     18205930.0         319402.0               55      2024-12-20                 4
  STR007     Toko Sejahtera Bogor     Bogor     Online           80     15701845.0         196273.0               76      2024-12-30                 4
  STR005        Toko Berkah Medan     Medan     Hybrid           63     15380280.0         244131.0               58      2024-12-14                 4
  STR050        Toko Makmur Medan     Medan     Online           51     14995290.0         294025.0               48      2024-12-23                 4
  STR042          Toko Maju Bogor     Bogor     Online           50     14414400.0         288288.0               48      2024-12-24                 4
  STR048     Toko Makmur Surabaya  Surabaya     Online           58     14412250.0         248

In [ ]:
# CELL 7 — Analisis 4: Pengaruh Diskon terhadap Volume
q4 = query("""
    SELECT
        discount_pct,
        COUNT(*)                             AS total_transaksi,
        ROUND(AVG(qty), 2)                   AS avg_qty,
        ROUND(AVG(total_price), 0)           AS avg_revenue_per_order,
        ROUND(SUM(total_price), 0)           AS total_revenue
    FROM transactions
    WHERE order_status = 'delivered'
    GROUP BY 1
    ORDER BY 1
""")
print(q4.to_string(index=False))
q4.to_csv('../data/processed/q4_discount_impact.csv', index=False)
print("\n✓ Disimpan")

 discount_pct  total_transaksi  avg_qty  avg_revenue_per_order  total_revenue
          0.0             1113     2.45               231662.0    257839300.0
          5.0              566     2.41               208690.0    118118630.0
         10.0              466     2.35               170149.0     79289460.0
         15.0              313     2.39               186656.0     58423220.0
         20.0              165     2.22               172787.0     28509920.0
         25.0               79     2.35               169403.0     13382850.0

✓ Disimpan


In [ ]:
# CELL 8 — Analisis 5: Cancellation & Return Rate per Kategori
q5 = query("""
    SELECT
        category,
        COUNT(*)                                                       AS total_order,
        SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END)   AS delivered,
        SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END)   AS cancelled,
        SUM(CASE WHEN order_status = 'returned'  THEN 1 ELSE 0 END)   AS returned,
        ROUND(
            SUM(CASE WHEN order_status = 'cancelled' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        )                                                              AS cancel_rate_pct,
        ROUND(
            SUM(CASE WHEN order_status = 'returned' THEN 1 ELSE 0 END)
            * 100.0 / COUNT(*), 2
        )                                                              AS return_rate_pct
    FROM transactions
    GROUP BY category
    ORDER BY cancel_rate_pct DESC
""")
print(q5.to_string(index=False))
q5.to_csv('../data/processed/q5_cancel_return_rate.csv', index=False)
print("\n✓ Disimpan")

                category  total_order  delivered  cancelled  returned  cancel_rate_pct  return_rate_pct
       Fashion & Pakaian          778      369.0      136.0     135.0            17.48            17.35
    Kerajinan & Souvenir          539      273.0       94.0      86.0            17.44            15.96
      Olahraga & Outdoor          470      226.0       81.0      77.0            17.23            16.38
  Kecantikan & Perawatan          763      386.0      129.0     118.0            16.91            15.47
            Rumah Tangga          821      398.0      138.0     151.0            16.81            18.39
   Elektronik & Aksesori          711      345.0      116.0     130.0            16.32            18.28
       Makanan & Minuman          770      415.0      124.0     105.0            16.10            13.64
Perlengkapan Bayi & Anak          562      290.0       90.0      84.0            16.01            14.95

✓ Disimpan


In [ ]:
# CELL 9 — Verifikasi Output & Tutup Koneksi

import os

files = [
    '../data/processed/q1_revenue_category_quarter.csv',
    '../data/processed/q2_payment_by_city.csv',
    '../data/processed/q3_store_performance.csv',
    '../data/processed/q4_discount_impact.csv',
    '../data/processed/q5_cancel_return_rate.csv',
]

print("OUTPUT FILES:")
for f in files:
    if os.path.exists(f):
        rows = len(pd.read_csv(f))
        print(f"  ✓ {os.path.basename(f):45s} {rows:3d} rows")
    else:
        print(f"  ✗ {os.path.basename(f)} — TIDAK DITEMUKAN")

con.close()
print("\nDuckDB connection closed ✓")

OUTPUT FILES:
  ✓ q1_revenue_category_quarter.csv                72 rows
  ✓ q2_payment_by_city.csv                        120 rows
  ✓ q3_store_performance.csv                       20 rows
  ✓ q4_discount_impact.csv                          6 rows
  ✓ q5_cancel_return_rate.csv                       8 rows

DuckDB connection closed ✓
